# Crystal Graph Convolutional Neural Networks (CGCNN)

## Introduction

This notebook implements the **Crystal Graph Convolutional Neural Network (CGCNN)** architecture, a deep learning framework designed for predicting material properties directly from crystal structures.

### Reference
**Paper**: Xie, T., & Grossman, J. C. (2018). "Crystal Graph Convolutional Neural Networks for an Accurate and Interpretable Prediction of Material Properties." *Physical Review Letters*, 120(14), 145301.

---

## Theoretical Background

### 1. Graph Representation of Crystals

In CGCNN, crystal structures are represented as **multigraphs** where:
- **Nodes (Vertices)**: Represent atoms in the crystal
- **Edges**: Represent bonds/interactions between atoms
- **Node Features**: Atom type, electron configuration, electronegativity, etc.
- **Edge Features**: Distance, bond type, coordination information

### 2. Convolution on Crystal Graphs

Unlike standard CNNs that operate on regular grids (images), crystal graphs have:
- Variable number of neighbors per atom
- Non-Euclidean structure
- Periodic boundary conditions

The convolutional operation aggregates information from neighboring atoms to update each atom's feature representation.

### 3. Message Passing Framework

The convolution layer implements a **message passing** scheme:

$$
\mathbf{v}_i^{(t+1)} = \mathbf{v}_i^{(t)} + \sum_{j \in \mathcal{N}(i)} \sigma\left(\mathbf{z}_{ij}^{(t)}\right) \odot \mathbf{g}\left(\mathbf{z}_{ij}^{(t)}\right)
$$

Where:
- $\mathbf{v}_i^{(t)}$: Feature vector of atom $i$ at layer $t$
- $\mathcal{N}(i)$: Set of neighboring atoms of atom $i$
- $\mathbf{z}_{ij}^{(t)} = \mathbf{v}_i^{(t)} \oplus \mathbf{v}_j^{(t)} \oplus \mathbf{u}_{ij}$: Concatenation of central atom, neighbor, and edge features
- $\sigma$: Sigmoid activation (gate function)
- $\mathbf{g}$: Core message function (typically with softplus activation)
- $\odot$: Element-wise multiplication

### 4. Architecture Components

#### a) **Embedding Layer**
Projects initial atom features to a learnable hidden representation:
$$
\mathbf{v}_i^{(0)} = \mathbf{W}_0 \mathbf{x}_i + \mathbf{b}_0
$$

#### b) **Convolutional Layers**
Multiple graph convolution layers that progressively refine atomic representations by aggregating local neighborhood information.

#### c) **Pooling Layer**
Aggregates all atomic features in a crystal to obtain a crystal-level representation:
$$
\mathbf{v}_{\text{crystal}} = \frac{1}{N} \sum_{i=1}^{N} \mathbf{v}_i^{(T)}
$$
(Mean pooling over all atoms)

#### d) **Fully Connected Layers**
Maps the pooled crystal representation to the target property (e.g., formation energy, band gap).

### 5. Key Innovations

1. **Gating Mechanism**: The sigmoid gate ($\sigma$) allows the network to learn which neighbor information is relevant
2. **Residual Connections**: Skip connections ($\mathbf{v}_i^{(t)} +$ ...) help with gradient flow
3. **Batch Normalization**: Stabilizes training and improves convergence
4. **Invariance to Atom Ordering**: Pooling ensures the prediction is invariant to the order of atoms

---

## Implementation

### Import Required Libraries

In [1]:
from __future__ import print_function, division

import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

PyTorch Version: 2.11.0+cpu
CUDA Available: False


---

## Convolutional Layer

### Theory: Graph Convolution Operation

The `ConvLayer` performs the core graph convolution operation. For each atom, it:

1. **Gathers neighbor features**: Retrieves features from all neighboring atoms
2. **Concatenates features**: Combines central atom, neighbor, and edge features
3. **Applies gating**: Uses a sigmoid gate to control information flow
4. **Aggregates**: Sums the gated messages from all neighbors
5. **Updates**: Adds the aggregated message to the original features (residual connection)

### Mathematical Formulation

Given:
- $\mathbf{v}_i \in \mathbb{R}^{d}$: Central atom feature (atom_fea_len)
- $\mathbf{v}_j \in \mathbb{R}^{d}$: Neighbor atom feature
- $\mathbf{u}_{ij} \in \mathbb{R}^{d'}$: Edge feature (nbr_fea_len)

The convolution operation:

$$
\mathbf{z}_{ij} = [\mathbf{v}_i \oplus \mathbf{v}_j \oplus \mathbf{u}_{ij}] \in \mathbb{R}^{2d + d'}
$$

$$
\mathbf{f}_{ij}, \mathbf{c}_{ij} = \text{split}\left(\text{BN}\left(\mathbf{W} \mathbf{z}_{ij} + \mathbf{b}\right)\right)
$$

$$
\mathbf{m}_{ij} = \sigma(\mathbf{f}_{ij}) \odot \text{softplus}(\mathbf{c}_{ij})
$$

$$
\mathbf{v}_i' = \text{softplus}\left(\mathbf{v}_i + \text{BN}\left(\sum_{j \in \mathcal{N}(i)} \mathbf{m}_{ij}\right)\right)
$$

In [2]:
class ConvLayer(nn.Module):
    """
    Convolutional operation on graphs
    
    This layer implements the graph convolution operation that aggregates
    information from neighboring atoms using a gating mechanism.
    """
    def __init__(self, atom_fea_len, nbr_fea_len):
        """
        Initialize ConvLayer.

        Parameters
        ----------
        atom_fea_len: int
          Number of atom hidden features.
        nbr_fea_len: int
          Number of bond features.
        """
        super(ConvLayer, self).__init__()
        self.atom_fea_len = atom_fea_len
        self.nbr_fea_len = nbr_fea_len
        
        # Fully connected layer: maps concatenated features to gated features
        # Input: [atom_i, atom_j, edge_ij] -> Output: [filter, core]
        self.fc_full = nn.Linear(2*self.atom_fea_len+self.nbr_fea_len,
                                 2*self.atom_fea_len)
        
        # Activation functions
        self.sigmoid = nn.Sigmoid()      # Gate function (0 to 1)
        self.softplus1 = nn.Softplus()   # Smooth approximation of ReLU
        self.softplus2 = nn.Softplus()
        
        # Batch normalization layers for training stability
        self.bn1 = nn.BatchNorm1d(2*self.atom_fea_len)
        self.bn2 = nn.BatchNorm1d(self.atom_fea_len)

    def forward(self, atom_in_fea, nbr_fea, nbr_fea_idx):
        """
        Forward pass

        N: Total number of atoms in the batch
        M: Max number of neighbors

        Parameters
        ----------
        atom_in_fea: Variable(torch.Tensor) shape (N, atom_fea_len)
          Atom hidden features before convolution
        nbr_fea: Variable(torch.Tensor) shape (N, M, nbr_fea_len)
          Bond features of each atom's M neighbors
        nbr_fea_idx: torch.LongTensor shape (N, M)
          Indices of M neighbors of each atom

        Returns
        -------
        atom_out_fea: nn.Variable shape (N, atom_fea_len)
          Atom hidden features after convolution
        """
        N, M = nbr_fea_idx.shape
        
        # Step 1: Gather neighbor features using indices
        # atom_nbr_fea: (N, M, atom_fea_len)
        atom_nbr_fea = atom_in_fea[nbr_fea_idx, :]
        
        # Step 2: Concatenate [central_atom, neighbor_atom, edge] features
        # total_nbr_fea: (N, M, 2*atom_fea_len + nbr_fea_len)
        total_nbr_fea = torch.cat(
            [atom_in_fea.unsqueeze(1).expand(N, M, self.atom_fea_len),
             atom_nbr_fea, nbr_fea], dim=2)
        
        # Step 3: Apply linear transformation and batch normalization
        total_gated_fea = self.fc_full(total_nbr_fea)
        total_gated_fea = self.bn1(total_gated_fea.view(
            -1, self.atom_fea_len*2)).view(N, M, self.atom_fea_len*2)
        
        # Step 4: Split into filter (gate) and core (message) components
        nbr_filter, nbr_core = total_gated_fea.chunk(2, dim=2)
        
        # Step 5: Apply activations
        nbr_filter = self.sigmoid(nbr_filter)     # Gate: controls information flow
        nbr_core = self.softplus1(nbr_core)       # Message: actual information
        
        # Step 6: Element-wise gating and sum over neighbors
        nbr_sumed = torch.sum(nbr_filter * nbr_core, dim=1)
        nbr_sumed = self.bn2(nbr_sumed)
        
        # Step 7: Residual connection + activation
        out = self.softplus2(atom_in_fea + nbr_sumed)
        
        return out

---

## Complete CGCNN Architecture

### Theory: End-to-End Network

The complete CGCNN architecture consists of:

1. **Embedding Layer**: Projects raw atom features to learnable representations
2. **Stacked Convolution Layers**: Multiple graph convolutions to capture multi-hop neighborhood information
3. **Pooling Layer**: Aggregates atom-level features to crystal-level representation
4. **Fully Connected Layers**: Maps crystal features to target properties
5. **Output Layer**: 
   - Regression: Single value (e.g., formation energy)
   - Classification: Log-softmax over classes (e.g., metal/non-metal)

### Network Flow

```
Input: (atom_features, edge_features, edge_indices, crystal_mapping)
   |
   v
[Embedding Layer] ──> atom_fea_len dimensions
   |
   v
[Conv Layer 1] ──────> Message passing iteration 1
   |
   v
[Conv Layer 2] ──────> Message passing iteration 2
   |
   v
   ...
   |
   v
[Conv Layer n] ──────> Message passing iteration n
   |
   v
[Pooling] ───────────> Mean over all atoms per crystal
   |
   v
[FC Layer 1] ────────> h_fea_len dimensions
   |
   v
[FC Layer 2] ────────> Optional additional hidden layers
   |
   v
[Output Layer] ──────> Prediction (1 for regression, 2+ for classification)
```

### Hyperparameters

- `orig_atom_fea_len`: Dimension of input atom features (e.g., 92 for one-hot encoding)
- `nbr_fea_len`: Dimension of edge features (e.g., distance-based Gaussian expansion)
- `atom_fea_len`: Hidden atom feature dimension (default: 64)
- `n_conv`: Number of convolution layers (default: 3)
- `h_fea_len`: Hidden dimension after pooling (default: 128)
- `n_h`: Number of fully connected layers (default: 1)

In [3]:
class CrystalGraphConvNet(nn.Module):
    """
    Create a crystal graph convolutional neural network for predicting total
    material properties.
    """
    def __init__(self, orig_atom_fea_len, nbr_fea_len,
                 atom_fea_len=64, n_conv=3, h_fea_len=128, n_h=1,
                 classification=False):
        """
        Initialize CrystalGraphConvNet.

        Parameters
        ----------
        orig_atom_fea_len: int
          Number of atom features in the input.
        nbr_fea_len: int
          Number of bond features.
        atom_fea_len: int
          Number of hidden atom features in the convolutional layers
        n_conv: int
          Number of convolutional layers
        h_fea_len: int
          Number of hidden features after pooling
        n_h: int
          Number of hidden layers after pooling
        classification: bool
          Whether this is a classification task
        """
        super(CrystalGraphConvNet, self).__init__()
        self.classification = classification
        
        # Embedding layer: projects input atom features to hidden space
        self.embedding = nn.Linear(orig_atom_fea_len, atom_fea_len)
        
        # Stack of convolutional layers
        self.convs = nn.ModuleList([ConvLayer(atom_fea_len=atom_fea_len,
                                    nbr_fea_len=nbr_fea_len)
                                    for _ in range(n_conv)])
        
        # Transition from convolutional to fully connected layers
        self.conv_to_fc = nn.Linear(atom_fea_len, h_fea_len)
        self.conv_to_fc_softplus = nn.Softplus()
        
        # Additional fully connected hidden layers
        if n_h > 1:
            self.fcs = nn.ModuleList([nn.Linear(h_fea_len, h_fea_len)
                                      for _ in range(n_h-1)])
            self.softpluses = nn.ModuleList([nn.Softplus()
                                             for _ in range(n_h-1)])
        
        # Output layer
        if self.classification:
            self.fc_out = nn.Linear(h_fea_len, 2)  # Binary classification
        else:
            self.fc_out = nn.Linear(h_fea_len, 1)  # Regression
        
        # Classification-specific components
        if self.classification:
            self.logsoftmax = nn.LogSoftmax(dim=1)
            self.dropout = nn.Dropout()

    def forward(self, atom_fea, nbr_fea, nbr_fea_idx, crystal_atom_idx):
        """
        Forward pass

        N: Total number of atoms in the batch
        M: Max number of neighbors
        N0: Total number of crystals in the batch

        Parameters
        ----------
        atom_fea: Variable(torch.Tensor) shape (N, orig_atom_fea_len)
          Atom features from atom type
        nbr_fea: Variable(torch.Tensor) shape (N, M, nbr_fea_len)
          Bond features of each atom's M neighbors
        nbr_fea_idx: torch.LongTensor shape (N, M)
          Indices of M neighbors of each atom
        crystal_atom_idx: list of torch.LongTensor of length N0
          Mapping from the crystal idx to atom idx

        Returns
        -------
        prediction: nn.Variable shape (N0, ) for regression or (N0, 2) for classification
          Predicted property for each crystal
        """
        # Step 1: Embed atom features
        atom_fea = self.embedding(atom_fea)
        
        # Step 2: Apply graph convolutions
        for conv_func in self.convs:
            atom_fea = conv_func(atom_fea, nbr_fea, nbr_fea_idx)
        
        # Step 3: Pool atom features to get crystal-level representation
        crys_fea = self.pooling(atom_fea, crystal_atom_idx)
        
        # Step 4: Apply fully connected layers
        crys_fea = self.conv_to_fc(self.conv_to_fc_softplus(crys_fea))
        crys_fea = self.conv_to_fc_softplus(crys_fea)
        
        # Apply dropout for classification
        if self.classification:
            crys_fea = self.dropout(crys_fea)
        
        # Additional hidden layers if specified
        if hasattr(self, 'fcs') and hasattr(self, 'softpluses'):
            for fc, softplus in zip(self.fcs, self.softpluses):
                crys_fea = softplus(fc(crys_fea))
        
        # Step 5: Output layer
        out = self.fc_out(crys_fea)
        
        # Apply log-softmax for classification
        if self.classification:
            out = self.logsoftmax(out)
        
        return out

    def pooling(self, atom_fea, crystal_atom_idx):
        """
        Pooling the atom features to crystal features

        N: Total number of atoms in the batch
        N0: Total number of crystals in the batch

        Parameters
        ----------
        atom_fea: Variable(torch.Tensor) shape (N, atom_fea_len)
          Atom feature vectors of the batch
        crystal_atom_idx: list of torch.LongTensor of length N0
          Mapping from the crystal idx to atom idx
          
        Returns
        -------
        crys_fea: Variable(torch.Tensor) shape (N0, atom_fea_len)
          Crystal feature vectors
        """
        # Verify that all atoms are accounted for
        assert sum([len(idx_map) for idx_map in crystal_atom_idx]) == \
            atom_fea.data.shape[0]
        
        # Average pooling: mean of all atom features in each crystal
        summed_fea = [torch.mean(atom_fea[idx_map], dim=0, keepdim=True)
                      for idx_map in crystal_atom_idx]
        
        return torch.cat(summed_fea, dim=0)

---

## Model Instantiation and Summary

Let's create example models for both regression and classification tasks.

In [4]:
# Example configuration for regression (e.g., formation energy prediction)
print("=" * 80)
print("REGRESSION MODEL: Predicting Formation Energy")
print("=" * 80)

model_regression = CrystalGraphConvNet(
    orig_atom_fea_len=92,    # Atom feature dimension (e.g., from atom_init.json)
    nbr_fea_len=41,          # Edge feature dimension (e.g., Gaussian expansion)
    atom_fea_len=64,         # Hidden atom features
    n_conv=3,                # 3 graph convolution layers
    h_fea_len=128,           # Hidden features after pooling
    n_h=1,                   # 1 hidden layer after pooling
    classification=False     # Regression task
)

print(model_regression)
print(f"\nTotal parameters: {sum(p.numel() for p in model_regression.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model_regression.parameters() if p.requires_grad):,}")

REGRESSION MODEL: Predicting Formation Energy
CrystalGraphConvNet(
  (embedding): Linear(in_features=92, out_features=64, bias=True)
  (convs): ModuleList(
    (0-2): 3 x ConvLayer(
      (fc_full): Linear(in_features=169, out_features=128, bias=True)
      (sigmoid): Sigmoid()
      (softplus1): Softplus(beta=1.0, threshold=20.0)
      (softplus2): Softplus(beta=1.0, threshold=20.0)
      (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (conv_to_fc): Linear(in_features=64, out_features=128, bias=True)
  (conv_to_fc_softplus): Softplus(beta=1.0, threshold=20.0)
  (fc_out): Linear(in_features=128, out_features=1, bias=True)
)

Total parameters: 80,833
Trainable parameters: 80,833


In [5]:
# Example configuration for classification (e.g., metal vs. non-metal)
print("=" * 80)
print("CLASSIFICATION MODEL: Metal vs. Non-Metal Classification")
print("=" * 80)

model_classification = CrystalGraphConvNet(
    orig_atom_fea_len=92,
    nbr_fea_len=41,
    atom_fea_len=64,
    n_conv=3,
    h_fea_len=128,
    n_h=1,
    classification=True      # Classification task
)

print(model_classification)
print(f"\nTotal parameters: {sum(p.numel() for p in model_classification.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model_classification.parameters() if p.requires_grad):,}")

CLASSIFICATION MODEL: Metal vs. Non-Metal Classification
CrystalGraphConvNet(
  (embedding): Linear(in_features=92, out_features=64, bias=True)
  (convs): ModuleList(
    (0-2): 3 x ConvLayer(
      (fc_full): Linear(in_features=169, out_features=128, bias=True)
      (sigmoid): Sigmoid()
      (softplus1): Softplus(beta=1.0, threshold=20.0)
      (softplus2): Softplus(beta=1.0, threshold=20.0)
      (bn1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (conv_to_fc): Linear(in_features=64, out_features=128, bias=True)
  (conv_to_fc_softplus): Softplus(beta=1.0, threshold=20.0)
  (fc_out): Linear(in_features=128, out_features=2, bias=True)
  (logsoftmax): LogSoftmax(dim=1)
  (dropout): Dropout(p=0.5, inplace=False)
)

Total parameters: 80,962
Trainable parameters: 80,962


---

## Model Testing with Dummy Data

Let's verify the model works correctly with synthetic input data.

In [6]:
# Create dummy data for testing
batch_size = 2           # Number of crystals in batch
n_atoms_1 = 8            # Number of atoms in crystal 1
n_atoms_2 = 6            # Number of atoms in crystal 2
total_atoms = n_atoms_1 + n_atoms_2
max_neighbors = 12       # Maximum number of neighbors per atom

# Atom features: (total_atoms, orig_atom_fea_len)
atom_fea = torch.randn(total_atoms, 92)

# Neighbor features (edge features): (total_atoms, max_neighbors, nbr_fea_len)
nbr_fea = torch.randn(total_atoms, max_neighbors, 41)

# Neighbor indices: (total_atoms, max_neighbors)
# Random indices pointing to neighbor atoms
nbr_fea_idx = torch.randint(0, total_atoms, (total_atoms, max_neighbors))

# Crystal-to-atom mapping: list of tensors indicating which atoms belong to which crystal
crystal_atom_idx = [
    torch.LongTensor(list(range(0, n_atoms_1))),           # Crystal 1: atoms 0-7
    torch.LongTensor(list(range(n_atoms_1, total_atoms)))  # Crystal 2: atoms 8-13
]

print("Input shapes:")
print(f"  atom_fea: {atom_fea.shape}")
print(f"  nbr_fea: {nbr_fea.shape}")
print(f"  nbr_fea_idx: {nbr_fea_idx.shape}")
print(f"  crystal_atom_idx: {len(crystal_atom_idx)} crystals")
print(f"    Crystal 0: {len(crystal_atom_idx[0])} atoms")
print(f"    Crystal 1: {len(crystal_atom_idx[1])} atoms")

Input shapes:
  atom_fea: torch.Size([14, 92])
  nbr_fea: torch.Size([14, 12, 41])
  nbr_fea_idx: torch.Size([14, 12])
  crystal_atom_idx: 2 crystals
    Crystal 0: 8 atoms
    Crystal 1: 6 atoms


In [7]:
# Test regression model
print("\n" + "=" * 80)
print("TESTING REGRESSION MODEL")
print("=" * 80)

model_regression.eval()
with torch.no_grad():
    output_regression = model_regression(atom_fea, nbr_fea, nbr_fea_idx, crystal_atom_idx)

print(f"\nOutput shape: {output_regression.shape}")
print(f"Output values (predicted properties):")
print(output_regression)
print(f"\nExpected: ({batch_size}, 1) for regression")
print(f"Actual: {output_regression.shape}")
print(f"✓ Test passed!" if output_regression.shape == (batch_size, 1) else "✗ Test failed!")


TESTING REGRESSION MODEL

Output shape: torch.Size([2, 1])
Output values (predicted properties):
tensor([[-10.0440],
        [ -9.8518]])

Expected: (2, 1) for regression
Actual: torch.Size([2, 1])
✓ Test passed!


In [8]:
# Test classification model
print("\n" + "=" * 80)
print("TESTING CLASSIFICATION MODEL")
print("=" * 80)

model_classification.eval()
with torch.no_grad():
    output_classification = model_classification(atom_fea, nbr_fea, nbr_fea_idx, crystal_atom_idx)

print(f"\nOutput shape: {output_classification.shape}")
print(f"Output values (log probabilities):")
print(output_classification)
print(f"\nProbabilities (after exp):")
print(torch.exp(output_classification))
print(f"\nExpected: ({batch_size}, 2) for binary classification")
print(f"Actual: {output_classification.shape}")
print(f"✓ Test passed!" if output_classification.shape == (batch_size, 2) else "✗ Test failed!")


TESTING CLASSIFICATION MODEL

Output shape: torch.Size([2, 2])
Output values (log probabilities):
tensor([[-1.0371e-05, -1.1472e+01],
        [-8.9407e-06, -1.1622e+01]])

Probabilities (after exp):
tensor([[9.9999e-01, 1.0415e-05],
        [9.9999e-01, 8.9686e-06]])

Expected: (2, 2) for binary classification
Actual: torch.Size([2, 2])
✓ Test passed!


## Summary

We have successfully constructed the **Crystal Graph Convolutional Neural Network (CGCNN)**!

### Key Takeaways:
1. **`ConvLayer`**: Implements the custom graph convolution that combines central atom, neighbor atom, and edge distance features using a gating mechanism.
2. **`CrystalGraphConvNet`**: The full end-to-end architecture that applies multiple convolutions, pools the atom-level features into a crystal-level representation, and maps it to target properties.
3. **Flexibility**: The model seamlessly handles variable numbers of atoms and neighbors across different crystal structures, and can be configured for either regression or classification tasks.

### Next Steps
In the next notebook (**`07_cgcnn_training.ipynb`**), we will combine this model architecture with the data loaders built in Notebook 05 to define the loss functions, optimizers, and execute the actual training loop on real materials data.